# FactChecker AI — Extended Training on Colab (Free T4 GPU)

This notebook:
1. Pulls 5 additional fake-news datasets from HuggingFace (~200k+ new rows)
2. Merges them with your existing CSVs (upload optional)
3. Retrains TF-IDF + Calibrated Logistic Regression
4. Retrains the meta-decision model
5. Downloads `model.joblib`, `vectorizer.joblib`, `meta_model.joblib`
   → drop these into `backend/data/` to deploy

**Runtime: Runtime → Change runtime type → T4 GPU (optional, CPU works fine for TF-IDF)**

In [ ]:
# ── Install dependencies ──────────────────────────────────────
!pip install -q datasets scikit-learn pandas numpy joblib

In [ ]:
import pandas as pd
import numpy as np
from datasets import load_dataset
import warnings
warnings.filterwarnings('ignore')

frames = []

# ── Dataset 1: clmentbisaillon/fake-and-real-news-dataset (44k) ──────────────
print('Loading clmentbisaillon/fake-and-real-news-dataset...')
try:
    ds1 = load_dataset('clmentbisaillon/fake-and-real-news-dataset', split='train')
    df1 = ds1.to_pandas()
    # columns: title, text, subject, date, label (0=real, 1=fake)
    df1['combined'] = (df1['title'].fillna('') + ' ' + df1['text'].fillna('')).str.strip()
    df1['label'] = df1['label'].astype(int)
    frames.append(df1[['combined', 'label']])
    print(f'  ✅ {len(df1)} rows')
except Exception as e:
    print(f'  ⚠️  Failed: {e}')

# ── Dataset 2: ErfanMoosaviMonazzah/fake-news-detection-dataset-English (44k) ─
print('Loading ErfanMoosaviMonazzah/fake-news-detection-dataset-English...')
try:
    ds2 = load_dataset('ErfanMoosaviMonazzah/fake-news-detection-dataset-English', split='train')
    df2 = ds2.to_pandas()
    # columns: text, label (0=real, 1=fake)
    df2['combined'] = df2['text'].fillna('').str.strip()
    df2['label'] = df2['label'].astype(int)
    frames.append(df2[['combined', 'label']])
    print(f'  ✅ {len(df2)} rows')
except Exception as e:
    print(f'  ⚠️  Failed: {e}')

# ── Dataset 3: difraud/difraud (95k deceptive/non-deceptive) ─────────────────
print('Loading difraud/difraud...')
try:
    ds3 = load_dataset('difraud/difraud', split='train')
    df3 = ds3.to_pandas()
    # columns: text, label (1=deceptive/fake, 0=non-deceptive/real)
    df3['combined'] = df3['text'].fillna('').str.strip()
    df3['label'] = df3['label'].astype(int)
    frames.append(df3[['combined', 'label']])
    print(f'  ✅ {len(df3)} rows')
except Exception as e:
    print(f'  ⚠️  Failed: {e}')

# ── Dataset 4: newsmediabias/fake_news_elections_labelled_data ────────────────
print('Loading newsmediabias/fake_news_elections_labelled_data...')
try:
    ds4 = load_dataset('newsmediabias/fake_news_elections_labelled_data', split='train')
    df4 = ds4.to_pandas()
    # columns: text, label ('fake'/'real')
    df4['combined'] = df4['text'].fillna('').str.strip()
    df4['label'] = df4['label'].str.strip().str.lower().map({'fake': 1, 'real': 0})
    df4 = df4.dropna(subset=['label'])
    df4['label'] = df4['label'].astype(int)
    frames.append(df4[['combined', 'label']])
    print(f'  ✅ {len(df4)} rows')
except Exception as e:
    print(f'  ⚠️  Failed: {e}')

# ── Dataset 5: ucsbnlp/liar (12.8k PolitiFact statements) ────────────────────
print('Loading ucsbnlp/liar...')
try:
    ds5 = load_dataset('ucsbnlp/liar', split='train')
    df5 = ds5.to_pandas()
    # label: 0=pants-fire, 1=false, 2=barely-true, 3=half-true, 4=mostly-true, 5=true
    # Map: 0,1 → fake(1), 4,5 → real(0), drop 2,3 (ambiguous)
    df5['combined'] = df5['statement'].fillna('').str.strip()
    df5['label'] = df5['label'].map({0: 1, 1: 1, 2: None, 3: None, 4: 0, 5: 0})
    df5 = df5.dropna(subset=['label'])
    df5['label'] = df5['label'].astype(int)
    frames.append(df5[['combined', 'label']])
    print(f'  ✅ {len(df5)} rows (after dropping ambiguous labels)')
except Exception as e:
    print(f'  ⚠️  Failed: {e}')

print(f'\nTotal frames loaded: {len(frames)}')

In [ ]:
# ── Optional: upload your existing CSVs from local machine ───────────────────
# Uncomment and run this cell if you want to include your original training CSVs

# from google.colab import files
# uploaded = files.upload()  # upload Fake.csv, True.csv, fake_news_dataset_44k.csv, fake_news_dataset_20k.csv
#
# for fname in uploaded:
#     if fname in ('Fake.csv', 'True.csv'):
#         df = pd.read_csv(fname, usecols=['title', 'text'])
#         df['label'] = 1 if fname == 'Fake.csv' else 0
#         df['combined'] = (df['title'].fillna('') + ' ' + df['text'].fillna('')).str.strip()
#         frames.append(df[['combined', 'label']])
#         print(f'Uploaded {fname}: {len(df)} rows')
#     elif '44k' in fname:
#         df = pd.read_csv(fname, usecols=['text', 'label'])
#         df = df.rename(columns={'text': 'combined'})
#         df['label'] = df['label'].astype(int)
#         frames.append(df[['combined', 'label']])
#         print(f'Uploaded {fname}: {len(df)} rows')
#     elif '20k' in fname:
#         df = pd.read_csv(fname, usecols=['title', 'text', 'label'])
#         df['label'] = df['label'].str.strip().str.lower().map({'fake': 1, 'real': 0})
#         df = df.dropna(subset=['label'])
#         df['combined'] = (df['title'].fillna('') + ' ' + df['text'].fillna('')).str.strip()
#         frames.append(df[['combined', 'label']])
#         print(f'Uploaded {fname}: {len(df)} rows')

In [ ]:
# ── Merge, clean, deduplicate ─────────────────────────────────
df = pd.concat(frames, ignore_index=True)
df = df.dropna(subset=['combined', 'label'])
df = df[df['combined'].str.len() >= 30]
df = df[df['combined'].str.contains(r'[a-zA-Z]', regex=True)]
df['combined'] = df['combined'].str[:5000]
df = df.drop_duplicates(subset=['combined'])
df['label'] = df['label'].astype(int)

print(f'Total samples after merge + dedup + quality filter: {len(df):,}')
print(f'  Fake : {df["label"].sum():,}')
print(f'  Real : {(df["label"]==0).sum():,}')
print(f'  Balance: {df["label"].mean():.2%} fake')

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, brier_score_loss
import joblib

X_train, X_test, y_train, y_test = train_test_split(
    df['combined'], df['label'],
    test_size=0.1, random_state=42, stratify=df['label']
)

print(f'Train: {len(X_train):,}  |  Test: {len(X_test):,}')

# ── Vectorize ─────────────────────────────────────────────────
vectorizer = TfidfVectorizer(
    stop_words='english',
    max_features=75000,   # bumped from 50k
    ngram_range=(1, 2),
    sublinear_tf=True,
    min_df=2,
)
print('Fitting vectorizer...')
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec  = vectorizer.transform(X_test)

# ── Train base model ──────────────────────────────────────────
print('Training Logistic Regression...')
base_model = LogisticRegression(max_iter=1000, C=5.0, solver='lbfgs', n_jobs=-1)

# ── Calibrate ─────────────────────────────────────────────────
print('Calibrating with isotonic regression...')
model = CalibratedClassifierCV(base_model, method='isotonic', cv=3)
model.fit(X_train_vec, y_train)

# ── Evaluate ──────────────────────────────────────────────────
y_pred  = model.predict(X_test_vec)
y_proba = model.predict_proba(X_test_vec)[:, 1]
acc     = accuracy_score(y_test, y_pred)
brier   = brier_score_loss(y_test, y_proba)

print(f'\n🎯 Test accuracy : {acc:.4f}')
print(f'📉 Brier score   : {brier:.4f}  (lower = better calibration)')
print(classification_report(y_test, y_pred, target_names=['Real', 'Fake']))

# ── Save ──────────────────────────────────────────────────────
joblib.dump(model,      'model.joblib')
joblib.dump(vectorizer, 'vectorizer.joblib')
print('✅ Saved model.joblib and vectorizer.joblib')

In [ ]:
# ── Train meta-decision model ─────────────────────────────────
# Simulates the three pipeline scores on the test set to train the meta-model.
# In production the meta-model fuses real ML + AI + evidence scores.

print('Generating meta-model training features...')

# ML scores on test set (already have these)
ml_scores = y_proba

# Simulate AI scores: add calibrated noise around ground truth
rng = np.random.default_rng(42)
ai_scores = np.clip(
    y_test.values * 0.85 + (1 - y_test.values) * 0.15 + rng.normal(0, 0.12, len(y_test)),
    0.0, 1.0
)

# Simulate evidence scores: inverse of fake probability with noise
ev_scores = np.clip(
    (1 - y_test.values) * 0.80 + y_test.values * 0.20 + rng.normal(0, 0.15, len(y_test)),
    0.0, 1.0
)

X_meta = np.column_stack([ml_scores, ai_scores, ev_scores])
y_meta = y_test.values

X_meta_train, X_meta_test, y_meta_train, y_meta_test = train_test_split(
    X_meta, y_meta, test_size=0.2, random_state=42
)

meta_base  = LogisticRegression(max_iter=500, C=1.0)
meta_model = CalibratedClassifierCV(meta_base, method='isotonic', cv=3)
meta_model.fit(X_meta_train, y_meta_train)

meta_pred = meta_model.predict(X_meta_test)
print(f'Meta-model accuracy: {accuracy_score(y_meta_test, meta_pred):.4f}')

joblib.dump(meta_model, 'meta_model.joblib')
print('✅ Saved meta_model.joblib')

In [ ]:
# ── Save model version info ───────────────────────────────────
import json, datetime

version_info = {
    'version': datetime.datetime.utcnow().strftime('%Y%m%d_%H%M'),
    'train_samples': int(len(X_train)),
    'test_samples': int(len(X_test)),
    'accuracy': round(float(acc), 4),
    'brier_score': round(float(brier), 4),
    'tfidf_features': 75000,
    'datasets': [
        'clmentbisaillon/fake-and-real-news-dataset',
        'ErfanMoosaviMonazzah/fake-news-detection-dataset-English',
        'difraud/difraud',
        'newsmediabias/fake_news_elections_labelled_data',
        'ucsbnlp/liar',
    ]
}

with open('model_version.json', 'w') as f:
    json.dump(version_info, f, indent=2)

print(json.dumps(version_info, indent=2))

In [ ]:
# ── Download all artifacts ────────────────────────────────────
# Files to drop into backend/data/
from google.colab import files

for fname in ['model.joblib', 'vectorizer.joblib', 'meta_model.joblib', 'model_version.json']:
    files.download(fname)
    print(f'⬇️  Downloading {fname}')

print('\nDone! Drop these files into backend/data/ and redeploy.')